In [5]:
pip install fastapi

In [6]:
# ==============================================================================
# FASTAPI BASICS CRASH COURSE
# ==============================================================================

from typing import Optional
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field
import uvicorn

# Initialize FastAPI App
app = FastAPI(
    title="FastAPI Crash Course API",
    description="A foundational API demonstrating routes, path/query params, Pydantic, and CRUD operations.",
    version="1.0.0"
)

In [7]:
# ------------------------------------------------------------------------------
# 1. IN-MEMORY DATABASE (Mock Data)
# ------------------------------------------------------------------------------
items_db = {
    1: {"id": 1, "name": "Gaming Mouse", "price": 49.99, "in_stock": True},
    2: {"id": 2, "name": "Mechanical Keyboard", "price": 89.99, "in_stock": True},
    3: {"id": 3, "name": "4K Monitor", "price": 299.99, "in_stock": False},
}

In [8]:
# ------------------------------------------------------------------------------
# 2. PYDANTIC MODELS (Data Validation & Schemas)
# ------------------------------------------------------------------------------
class Item(BaseModel):
    name: str = Field(..., min_length=2, json_schema_extra={"example": "Wireless Headset"})
    price: float = Field(..., gt=0, json_schema_extra={"example": 59.99})
    in_stock: bool = True
    description: Optional[str] = None


class ItemUpdate(BaseModel):
    name: Optional[str] = None
    price: Optional[float] = None
    in_stock: Optional[bool] = None

/tmp/ipykernel_1147/2328783673.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  name: str = Field(..., min_length=2, example="Wireless Headset")
/tmp/ipykernel_1147/2328783673.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  price: float = Field(..., gt=0, example=59.99)


In [9]:
# ------------------------------------------------------------------------------
# 3. ROOT & BASIC GET ENDPOINTS
# ------------------------------------------------------------------------------
@app.get("/", tags=["General"])
def read_root():
    """Root endpoint returning a welcome message."""
    return {"message": "Welcome to the FastAPI Crash Course!", "docs": "/docs"}

In [10]:
# ------------------------------------------------------------------------------
# 4. PATH PARAMETERS & QUERY PARAMETERS
# ------------------------------------------------------------------------------
@app.get("/items", tags=["Items"])
def get_all_items(in_stock: Optional[bool] = None, limit: int = 10):
    """
    Get all items with optional query filtering:
    - /items -> returns all items up to limit
    - /items?in_stock=true -> filters items in stock
    """
    results = list(items_db.values())

    if in_stock is not None:
        results = [item for item in results if item["in_stock"] == in_stock]

    return results[:limit]


@app.get("/items/{item_id}", tags=["Items"])
def get_item_by_id(item_id: int):
    """
    Get a specific item by its integer ID (Path Parameter).
    Returns 404 HTTP Exception if not found.
    """
    if item_id not in items_db:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Item with ID {item_id} not found."
        )
    return items_db[item_id]

In [11]:
# ------------------------------------------------------------------------------
# 5. POST REQUEST (Creating Data with Validation)
# ------------------------------------------------------------------------------
@app.post("/items", status_code=status.HTTP_201_CREATED, tags=["Items"])
def create_item(item: Item):
    """
    Create a new item. FastAPI validates the JSON body automatically using Pydantic.
    """
    new_id = max(items_db.keys(), default=0) + 1
    new_item = {"id": new_id, **item.model_dump()}
    items_db[new_id] = new_item
    return new_item

In [12]:
# ------------------------------------------------------------------------------
# 6. PUT / PATCH REQUEST (Updating Data)
# ------------------------------------------------------------------------------
@app.put("/items/{item_id}", tags=["Items"])
def update_item(item_id: int, item_data: ItemUpdate):
    """
    Update an existing item by ID.
    """
    if item_id not in items_db:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Item with ID {item_id} not found."
        )

    stored_item = items_db[item_id]
    update_data = item_data.model_dump(exclude_unset=True)  # Only updated fields
    stored_item.update(update_data)
    items_db[item_id] = stored_item

    return stored_item

In [13]:
# ------------------------------------------------------------------------------
# 7. DELETE REQUEST
# ------------------------------------------------------------------------------
@app.delete("/items/{item_id}", status_code=status.HTTP_200_OK, tags=["Items"])
def delete_item(item_id: int):
    """
    Delete an item by ID.
    """
    if item_id not in items_db:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Item with ID {item_id} not found."
        )

    deleted_item = items_db.pop(item_id)
    return {"message": f"Item '{deleted_item['name']}' deleted successfully."}

In [ ]:
# ------------------------------------------------------------------------------
# 8. PROGRAMMATIC SERVER LAUNCH (Run directly in VS Code with F5 / Play button)
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    print("\n🚀 Starting FastAPI server with Uvicorn...")
    print("📍 Interactive Docs (Swagger UI): http://127.0.0.1:8000/docs")
    print("📍 Alternative Docs (ReDoc):     http://127.0.0.1:8000/redoc")
    print("----------------------------------------------------------\n")

    # Run the app directly in the notebook environment
    uvicorn.run(app, host="0.0.0.0", port=8000)

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [1147] using WatchFiles



🚀 Starting FastAPI server with Uvicorn...
📍 Interactive Docs (Swagger UI): http://127.0.0.1:8000/docs
📍 Alternative Docs (ReDoc):     http://127.0.0.1:8000/redoc
----------------------------------------------------------

